# Global Solution 2026 — Dynamic Programming

## Análise de resultados

Notebook para analisar os resultados dos algoritmos de Força Bruta e Dijkstra nos cenários brasileiros escolhidos.

**Cenários:**

- Enchentes no Rio Grande do Sul
- Seca no MATOPIBA


## 1. Importações


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from src.data_loader import carregar_cenario_processado
from src.data_structures import construir_bst_por_risco, criar_mapa_municipios
from src.greedy import dijkstra
from src.visualizations import (
    plotar_grafo,
    plotar_bst,
    plotar_desempenho,
    plotar_gap_otimalidade,
    plotar_memoria,
    plotar_operacoes,
    plotar_tabela_estruturas,
)


## 2. Leitura dos resultados

Antes de executar esta etapa, rode o projeto pela raiz:

```bash
python src/main.py
```


In [ ]:
resultados_rs_path = ROOT / "data" / "processed" / "resultados_rs.csv"
resultados_matopiba_path = ROOT / "data" / "processed" / "resultados_matopiba.csv"

df_rs = pd.read_csv(resultados_rs_path)
df_matopiba = pd.read_csv(resultados_matopiba_path)

df_rs["cenario_curto"] = "RS"
df_matopiba["cenario_curto"] = "MATOPIBA"

df = pd.concat([df_rs, df_matopiba], ignore_index=True)
df.head()


## 3. Visão geral dos dados de desempenho


In [ ]:
df.info()


In [ ]:
df.groupby(["cenario_curto", "algoritmo"])[
    ["tempo_ms", "memoria_mb", "operacoes", "custo_solucao"]
].agg(["mean", "min", "max"]).round(4)


## 4. Comparação de tempo de execução

A comparação abaixo mostra o crescimento do tempo de execução em função do número de vértices. A Força Bruta é limitada a instâncias pequenas porque enumera possibilidades de caminho, enquanto o Dijkstra usa fila de prioridade e escala melhor para grafos maiores.


In [ ]:
for cenario in df["cenario_curto"].unique():
    dados = df[df["cenario_curto"] == cenario]

    plt.figure(figsize=(10, 6))

    for algoritmo in dados["algoritmo"].unique():
        serie = dados[dados["algoritmo"] == algoritmo].sort_values("n_vertices")
        plt.plot(serie["n_vertices"], serie["tempo_ms"], marker="o", label=algoritmo)

    plt.title(f"Tempo de execução x N — {cenario}")
    plt.xlabel("Número de vértices")
    plt.ylabel("Tempo de execução (ms)")
    plt.legend()
    plt.grid(True)
    plt.show()


## 5. Comparação de memória


In [ ]:
for cenario in df["cenario_curto"].unique():
    dados = df[df["cenario_curto"] == cenario]

    plt.figure(figsize=(10, 6))

    for algoritmo in dados["algoritmo"].unique():
        serie = dados[dados["algoritmo"] == algoritmo].sort_values("n_vertices")
        plt.plot(serie["n_vertices"], serie["memoria_mb"], marker="o", label=algoritmo)

    plt.title(f"Memória alocada x N — {cenario}")
    plt.xlabel("Número de vértices")
    plt.ylabel("Memória alocada (MB)")
    plt.legend()
    plt.grid(True)
    plt.show()


## 6. Comparação de operações elementares

Para Força Bruta, o contador representa chamadas recursivas. Para Dijkstra, representa arestas relaxadas.


In [ ]:
for cenario in df["cenario_curto"].unique():
    dados = df[df["cenario_curto"] == cenario]

    plt.figure(figsize=(10, 6))

    for algoritmo in dados["algoritmo"].unique():
        serie = dados[dados["algoritmo"] == algoritmo].sort_values("n_vertices")
        plt.plot(serie["n_vertices"], serie["operacoes"], marker="o", label=algoritmo)

    plt.title(f"Operações elementares x N — {cenario}")
    plt.xlabel("Número de vértices")
    plt.ylabel("Operações elementares")
    plt.legend()
    plt.grid(True)
    plt.show()


## 7. Gap de otimalidade

O gap compara a solução gulosa com a solução ótima obtida por Força Bruta nas instâncias pequenas em que os dois algoritmos foram executados.

\[
gap = 
rac{custo\_guloso - custo\_otimo}{custo\_otimo} 	imes 100
\]


In [ ]:
def calcular_gap_df(dados):
    linhas = []

    for n in sorted(dados["n_vertices"].unique()):
        dados_n = dados[dados["n_vertices"] == n]

        fb = dados_n[dados_n["algoritmo"].str.contains("Força|Bruta", case=False, regex=True)]
        guloso = dados_n[dados_n["algoritmo"].str.contains("Dijkstra|Guloso", case=False, regex=True)]

        if fb.empty or guloso.empty:
            continue

        custo_otimo = float(fb.iloc[0]["custo_solucao"])
        custo_guloso = float(guloso.iloc[0]["custo_solucao"])

        gap = 0.0 if custo_otimo <= 0 else ((custo_guloso - custo_otimo) / custo_otimo) * 100

        linhas.append({
            "n_vertices": n,
            "custo_otimo": custo_otimo,
            "custo_guloso": custo_guloso,
            "gap_percentual": gap,
        })

    return pd.DataFrame(linhas)


gaps = []

for cenario in df["cenario_curto"].unique():
    gap_cenario = calcular_gap_df(df[df["cenario_curto"] == cenario])
    gap_cenario["cenario_curto"] = cenario
    gaps.append(gap_cenario)

df_gap = pd.concat(gaps, ignore_index=True)
df_gap


In [ ]:
for cenario in df_gap["cenario_curto"].unique():
    dados = df_gap[df_gap["cenario_curto"] == cenario]

    plt.figure(figsize=(10, 6))
    plt.plot(dados["n_vertices"], dados["gap_percentual"], marker="o")
    plt.title(f"Gap de otimalidade x N — {cenario}")
    plt.xlabel("Número de vértices")
    plt.ylabel("Gap de otimalidade (%)")
    plt.grid(True)
    plt.show()


## 8. Carregamento dos grafos e das BSTs


In [ ]:
municipios_rs, grafo_rs = carregar_cenario_processado(
    ROOT / "data" / "processed" / "municipios_rs.json",
    ROOT / "data" / "processed" / "grafo_rs.json"
)

municipios_matopiba, grafo_matopiba = carregar_cenario_processado(
    ROOT / "data" / "processed" / "municipios_matopiba.json",
    ROOT / "data" / "processed" / "grafo_matopiba.json"
)

bst_rs = construir_bst_por_risco(municipios_rs[:15])
bst_matopiba = construir_bst_por_risco(municipios_matopiba[:15])

mapa_rs = criar_mapa_municipios(municipios_rs)
mapa_matopiba = criar_mapa_municipios(municipios_matopiba)

len(municipios_rs), len(grafo_rs), len(municipios_matopiba), len(grafo_matopiba)


## 9. Visualização das rotas encontradas pelo Dijkstra


In [ ]:
origem_rs = municipios_rs[0][0]
destino_rs = municipios_rs[-1][0]
rota_rs = dijkstra(grafo_rs, origem_rs, destino_rs)

origem_matopiba = municipios_matopiba[0][0]
destino_matopiba = municipios_matopiba[-1][0]
rota_matopiba = dijkstra(grafo_matopiba, origem_matopiba, destino_matopiba)

rota_rs.caminho, rota_rs.custo_total, rota_matopiba.caminho, rota_matopiba.custo_total


In [ ]:
plotar_grafo(
    grafo_rs,
    municipios=mapa_rs,
    caminho_destaque=rota_rs.caminho,
    titulo="Grafo de municípios — RS com rota destacada"
)


In [ ]:
plotar_grafo(
    grafo_matopiba,
    municipios=mapa_matopiba,
    caminho_destaque=rota_matopiba.caminho,
    titulo="Grafo de municípios — MATOPIBA com rota destacada"
)


## 10. Visualização da BST

As BSTs abaixo usam uma amostra de 15 municípios para manter a visualização legível.


In [ ]:
plotar_bst(
    bst_rs,
    titulo="BST por risco — RS"
)


In [ ]:
plotar_bst(
    bst_matopiba,
    titulo="BST por risco — MATOPIBA"
)


## 11. Municípios de maior risco


In [ ]:
bst_rs_completa = construir_bst_por_risco(municipios_rs)
bst_matopiba_completa = construir_bst_por_risco(municipios_matopiba)

alto_risco_rs = bst_rs_completa.buscar_intervalo(0.70, 1.00)
alto_risco_matopiba = bst_matopiba_completa.buscar_intervalo(0.70, 1.00)

df_alto_risco_rs = pd.DataFrame(
    alto_risco_rs,
    columns=["id_municipio", "nome", "indice_risco", "custo_atendimento", "populacao"]
).sort_values("indice_risco", ascending=False)

df_alto_risco_matopiba = pd.DataFrame(
    alto_risco_matopiba,
    columns=["id_municipio", "nome", "indice_risco", "custo_atendimento", "populacao"]
).sort_values("indice_risco", ascending=False)

df_alto_risco_rs.head(10)


In [ ]:
df_alto_risco_matopiba.head(10)


## 12. Tabela das estruturas de dados utilizadas


In [ ]:
estruturas = pd.DataFrame(
    [
        ["list", "Lista de adjacência, caminhos e resultados"],
        ["tuple", "Municípios e arestas"],
        ["dict", "Grafo, distâncias, predecessores e metadados"],
        ["set", "Visitados e prevenção de ciclos"],
        ["heapq", "Fila de prioridade do Dijkstra"],
        ["BST", "Organização por índice de risco"],
        ["Grafo", "Rede ponderada de municípios"],
    ],
    columns=["Estrutura", "Uso no sistema"]
)

estruturas


## 13. Escala de decisão

A escala abaixo classifica a solução considerando qualidade, custo computacional e aplicabilidade prática.


In [ ]:
escala_decisao = pd.DataFrame(
    [
        ["Excelente", "Gap próximo de 0%, baixo tempo e baixa memória", "Uso recomendado em cenários reais"],
        ["Bom", "Pequeno aumento de custo, mas execução ainda eficiente", "Uso recomendado com monitoramento"],
        ["Regular", "Tempo ou memória começam a crescer de forma relevante", "Uso restrito a instâncias médias"],
        ["Inviável", "Força Bruta cresce de forma combinatória", "Não recomendado para operação real"],
    ],
    columns=["Nível", "Critério", "Decisão prática"]
)

escala_decisao


## 14. Conclusão

A Força Bruta é útil como baseline para validar a solução ótima em instâncias pequenas, mas se torna inviável conforme o número de vértices aumenta. O Dijkstra apresentou melhor escalabilidade e é mais adequado para cenários reais de monitoramento ambiental, principalmente quando combinado com lista de adjacência, dicionários de distância, predecessores e heap como fila de prioridade.

A BST complementa a solução ao permitir a priorização dos municípios por índice de risco, conectando a modelagem de dados à tomada de decisão operacional em resposta a desastres ambientais.
